# EEG-Based Neurological Condition Classification: AD vs CN

**PharmaHacks 2026** — Team mlers

This notebook presents a complete pipeline for classifying Alzheimer's Disease (AD) vs Cognitively Normal (CN) subjects using EEG data. We employ two fundamentally different approaches — a dual-branch CNN+Transformer (DICE-net) and XGBoost — and combine them in a meta-ensemble.

## Table of Contents
1. Data Exploration
2. Feature Engineering
3. Model Architectures
4. Validation Strategy (LOOCV)
5. Results & Ensemble
6. Interpretability

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

import src.config as cfg
from src.cache import load_manifest

print(f'Device: {cfg.DEVICE}')
print(f'Data root: {cfg.DATA_ROOT}')

## 1. Data Exploration

The dataset contains 19-channel EEG recordings from 38 subjects (25 AD, 13 CN) sampled at 500 Hz. Recordings vary in length from ~30s to ~21 minutes.

In [ ]:
# Load label mapping
df = pd.read_csv(cfg.LABEL_CSV)
df_ac = df[df['label'].isin(['A', 'C'])].copy()
df_ac['condition'] = df_ac['label'].map({'A': 'AD', 'C': 'CN'})

print(f'Total subjects: {len(df_ac)}')
print(f'Class distribution:')
print(df_ac['condition'].value_counts().to_string())

# Show recording lengths
lengths = []
for _, row in df_ac.iterrows():
    sid = str(row['anonymized_id'])
    class_dir = cfg.CLASS_DIR[row['label']]
    npy_path = cfg.DATA_ROOT / class_dir / f'{sid}.npy'
    if npy_path.exists():
        eeg = np.load(npy_path)
        lengths.append({'subject': sid, 'condition': row['label'],
                       'samples': eeg.shape[1], 
                       'duration_sec': eeg.shape[1] / cfg.SFREQ})

len_df = pd.DataFrame(lengths)
print(f'\nRecording durations (seconds):')
print(len_df.groupby('condition')['duration_sec'].describe().round(1))

In [ ]:
# Visualize recording length distribution
fig, ax = plt.subplots(figsize=(10, 4))
ad_lens = len_df[len_df['condition'] == 'A']['duration_sec']
cn_lens = len_df[len_df['condition'] == 'C']['duration_sec']
ax.hist(ad_lens, bins=15, alpha=0.7, label=f'AD (n={len(ad_lens)})', color='#EF5350')
ax.hist(cn_lens, bins=15, alpha=0.7, label=f'CN (n={len(cn_lens)})', color='#42A5F5')
ax.set_xlabel('Recording Duration (seconds)')
ax.set_ylabel('Count')
ax.set_title('EEG Recording Duration Distribution')
ax.legend()
plt.tight_layout()
plt.savefig('../output/recording_durations.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Feature Engineering

Each recording is:
1. **Z-scored** per channel (zero mean, unit variance)
2. **Segmented** into 30-second epochs with 50% overlap (stride=15s)
3. **Feature extracted** per epoch:
   - **RBP** (Relative Band Power): Welch PSD across 5 frequency bands (delta, theta, alpha, beta, gamma) for each of 30 one-second sub-windows × 19 channels → shape `(30, 5, 19)`
   - **PLV** (Phase Locking Value): Average inter-channel phase coherence via CWT at 5 center frequencies for each sub-window × channel → shape `(30, 5, 19)`

**Why two features?** RBP captures spectral power distribution (how much energy in each frequency band). PLV captures functional connectivity (how synchronized channels are). These provide complementary information — power vs. coordination.

In [ ]:
# Show feature shapes from cache
manifest = load_manifest()
example_sid = list(manifest.keys())[0]
rbp = np.load(cfg.CACHE_DIR / f'{example_sid}_rbp.npy')
plv = np.load(cfg.CACHE_DIR / f'{example_sid}_scc.npy')

print(f'Subject {example_sid} ({manifest[example_sid]["label"]})')
print(f'  RBP shape: {rbp.shape}  (epochs, time_windows, bands, channels)')
print(f'  PLV shape: {plv.shape}')
print(f'  Epochs: {rbp.shape[0]} (with 50% overlap)')
print(f'\nRBP value range: [{rbp.min():.4f}, {rbp.max():.4f}]')
print(f'PLV value range: [{plv.min():.4f}, {plv.max():.4f}]')
print(f'\nFrequency bands: {cfg.FREQ_BANDS}')
print(f'Morlet center frequencies: {cfg.MORLET_FREQS} Hz')

## 3. Model Architectures

### DICE-net (Dual-Input Convolutional Encoder + Transformer)

```
RBP (B,30,5,19) ──► CNNBranch ──► (B,30,1024) ──┐
                                                   ├──► concat (B,30,2048)
PLV (B,30,5,19) ──► CNNBranch ──► (B,30,1024) ──┘       │
                                                    Linear(2048→128)
                                                         │
                                                   + pos_embed
                                                         │
                                               TransformerEncoder
                                               (d=128, h=4, L=2)
                                                         │
                                                    global avg pool
                                                         │
                                              Dropout→64→ReLU→Dropout→1
                                                         │
                                                    logit (sigmoid for P(AD))
```

**CNNBranch**: Conv3d(1→32→64→128) with MaxPool3d + AvgPool3d to reduce spatial dims.

### XGBoost (Gradient Boosted Trees)

- 418 handcrafted features per 30s window at 128Hz
- Features: band power, band ratios, Hjorth parameters, spectral entropy, statistical moments
- XGBClassifier: depth=4, lr=0.03, subsample=0.6, strong L2 regularization

### Why both complement each other

- DICE-net learns spatial-temporal patterns directly from structured features (30×5×19 tensors)
- XGBoost operates on flattened handcrafted features with explicit domain knowledge
- Different feature spaces + different model families = different error patterns

In [ ]:
# Model parameter count
from src.model import DICENet
import torch

model = DICENet()
n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'DICE-net parameters: {n_params:,} total, {n_trainable:,} trainable')
print(f'\nArchitecture:')
print(f'  CNN filters: {cfg.CNN_FILTERS}')
print(f'  Transformer: d_model={cfg.D_MODEL}, heads={cfg.N_HEADS}, layers={cfg.N_LAYERS}')
print(f'  Branch dim: {cfg.BRANCH_DIM} → fused: {cfg.FUSED_DIM}')

## 4. Validation Strategy: Leave-One-Out Cross-Validation (LOOCV)

With only 38 subjects, LOOCV is the gold standard:
- Train on 37 subjects, evaluate on 1 held-out subject
- Repeat 38 times
- Subject-level prediction: average P(AD) across all epochs of the held-out subject

**Training details:**
- Loss: BCEWithLogitsLoss with class-imbalance weighting (pos_weight = n_CN/n_AD)
- Label smoothing: ε=0.05
- Optimizer: AdamW (lr=5e-4, weight_decay=1e-4)
- Scheduler: Cosine annealing to 1e-5
- Early stopping: patience=20 epochs
- Augmentation: Gaussian noise, amplitude scaling, window/channel/band dropout

**Note on early stopping:** We early-stop on the held-out subject's loss, which creates a mild optimistic bias in LOOCV estimates. The actual test performance is evaluated on 14 separate held-out subjects not used in any training.

In [ ]:
# Load LOOCV results (pre-computed)
import csv

output_dir = Path('../output')

def load_loocv(path):
    results = []
    with open(path) as f:
        for row in csv.DictReader(f):
            results.append(row)
    return results

dice_path = output_dir / 'dicenet_loocv.csv'
xgb_path = Path('../xgboost_predictions/loocv.csv')

if dice_path.exists():
    dice_results = load_loocv(dice_path)
    dice_correct = sum(1 for r in dice_results if r['true_label'] == r['pred_label'])
    print(f'DICE-net LOOCV: {dice_correct}/{len(dice_results)} = {dice_correct/len(dice_results):.1%}')
else:
    print(f'DICE-net LOOCV results not found. Run: python scripts/predict.py')

if xgb_path.exists():
    xgb_results = load_loocv(xgb_path)
    xgb_correct = sum(1 for r in xgb_results if r['correct'] == 'True')
    print(f'XGBoost LOOCV:  {xgb_correct}/{len(xgb_results)} = {xgb_correct/len(xgb_results):.1%}')
else:
    print(f'XGBoost LOOCV results not found.')

## 5. Results & Ensemble

### Multi-Seed DICE-net Ensemble

Train 3-5 final models with different random seeds and average their predicted probabilities. This reduces variance from random initialization.

### Meta-Ensemble

Combine DICE-net and XGBoost predictions via weighted average:

P_ensemble(AD) = w × P_DICE(AD) + (1-w) × P_XGB(AD)

The optimal weight `w` is found by grid search on LOOCV out-of-fold predictions.

In [ ]:
# Load ensemble results if available
ens_path = output_dir / 'ensemble_loocv.csv'
if ens_path.exists():
    ens_results = load_loocv(ens_path)
    ens_correct = sum(1 for r in ens_results if r['correct'] == 'True')
    dice_w = ens_results[0].get('dice_weight', '?')
    print(f'Meta-Ensemble LOOCV: {ens_correct}/{len(ens_results)} = {ens_correct/len(ens_results):.1%}')
    print(f'Optimal weight: DICE={dice_w}')
else:
    print('Ensemble results not found. Run: python scripts/ensemble.py')

# Load test predictions
pred_path = Path('../predictions.csv')
if pred_path.exists():
    test_preds = load_loocv(pred_path)
    n_ad = sum(1 for r in test_preds if r['predicted_label'] == 'A')
    n_cn = len(test_preds) - n_ad
    print(f'\nTest predictions: {len(test_preds)} subjects')
    print(f'  Predicted AD: {n_ad}, Predicted CN: {n_cn}')
else:
    print('\nTest predictions not found. Run: python scripts/predict.py')

## 6. Interpretability

All plots are generated by `scripts/visualize.py` and saved to `output/`.

In [ ]:
# Display pre-generated plots
from IPython.display import Image, display

plots = [
    ('Confusion Matrices', 'confusion_matrices.png'),
    ('ROC Curves', 'roc_curves.png'),
    ('Per-Subject Confidence', 'subject_confidence.png'),
    ('Model Agreement', 'model_agreement.png'),
    ('Frequency Band Power (AD vs CN)', 'band_power.png'),
]

for title, filename in plots:
    path = output_dir / filename
    if path.exists():
        print(f'\n### {title}')
        display(Image(filename=str(path), width=800))
    else:
        print(f'\n### {title} — not generated yet. Run: python scripts/visualize.py')

## Key Insights

1. **Two complementary approaches**: DICE-net learns spatial-temporal patterns from structured EEG features; XGBoost uses handcrafted domain features. Their error patterns differ.

2. **PLV > spectral magnitude**: Replacing redundant CWT magnitude with Phase Locking Value connectivity gives the model genuinely different information from RBP.

3. **LOOCV is the right validation**: With 38 subjects, k-fold CV wastes data. LOOCV maximizes training data per fold.

4. **Multi-seed ensemble reduces variance**: Averaging predictions across 3-5 random seeds reduces sensitivity to initialization.

5. **Frequency band differences**: AD subjects show altered power distribution across canonical EEG bands, consistent with literature on AD-related slowing.

## Reproducibility

```bash
pip install -r requirements.txt
python scripts/precompute_features.py   # ~5-15 min
python scripts/predict.py               # LOOCV + multi-seed ensemble
python scripts/ensemble.py              # Meta-ensemble analysis
python scripts/visualize.py             # Generate all plots
```